# 03 — Modelloppsett

Dette notebooket klargjør felles modellgrunnlag for notebook `03a`, `03b` og `03c`.
Det gjør ingen modelltrening.

**Input:** `intermediate/df_processed.parquet`

**Output:** `intermediate/df_iso.parquet`, `intermediate/fill_weekly_median.pkl`


In [1]:
import os
import pandas as pd

from src.config import INTERMEDIATE_DIR, TRAIN_YEARS, TEST_YEARS, apply_style
from src.model_training import prepare_iso_data, save_prepared_data

apply_style()
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)


In [2]:
df = pd.read_parquet(f"{INTERMEDIATE_DIR}df_processed.parquet")
df_iso, weekly_median = prepare_iso_data(df, train_years=TRAIN_YEARS)

train_mask = df_iso.index.year.isin(TRAIN_YEARS)
test_mask = df_iso.index.year.isin(TEST_YEARS)

print(f"Totalt antall timer: {len(df):,}")
print(f"ISO-timer:          {len(df_iso):,} ({100 * len(df_iso) / len(df):.1f} %)")
print(f"Treningsperiode:    {TRAIN_YEARS} -> {train_mask.sum():,} timer")
print(f"Testperiode:        {TEST_YEARS} -> {test_mask.sum():,} timer")


Totalt antall timer: 53,394
ISO-timer:          34,916 (65.4 %)
Treningsperiode:    [2020, 2021, 2022, 2023] -> 25,072 timer
Testperiode:        [2024, 2025, 2026] -> 9,844 timer


In [3]:
summary_cols = ["price_NO4", "cons_NO4", "fill_avvik", "prod_wind_NO4"]
display(df_iso[summary_cols].describe().round(2))

print("Ukesmedianer for fyllingsgrad (treningsår):")
display(weekly_median.round(3).to_frame("median_fill_rate").head())


,price_NO4,cons_NO4,fill_avvik,prod_wind_NO4
count,34916.00,34916.00,34916.00,34916.00
mean,270.72,2276.77,0.00,301.46
std,292.61,410.81,0.10,210.33
min,-275.75,888.50,-0.22,0.00
25%,103.63,1917.00,-0.04,134.00
50%,207.16,2281.00,0.00,256.00
75%,363.00,2616.00,0.06,437.00
max,5335.09,4211.00,0.30,1030.34


Ukesmedianer for fyllingsgrad (treningsår):


,median_fill_rate
week,
1,0.670
2,0.653
3,0.651
4,0.638
5,0.614


In [4]:
save_prepared_data(df_iso, weekly_median, intermediate_dir=INTERMEDIATE_DIR)
print("Lagret modellgrunnlag:")
print(f"  {INTERMEDIATE_DIR}df_iso.parquet")
print(f"  {INTERMEDIATE_DIR}fill_weekly_median.pkl")


Lagret modellgrunnlag:
  intermediate/df_iso.parquet
  intermediate/fill_weekly_median.pkl
